# Part 3 — AutoGluon End-to-End ML with Metrics

This notebook demonstrates one complete machine-learning workflow using AutoGluon:

**prepare data → build a baseline → train AutoGluon → compare models → evaluate with multiple metrics → test decision thresholds → inspect feature importance**

The example predicts:

**`overspent_next` = whether a household is likely to overspend next month.**

The notebook keeps the dataset and model set deliberately small so the full workflow can run reliably in Google Colab.


## 1. Setup and Create the Dataset

### What we saw before
- This is the starting point of the end-to-end ML workflow.
- We first need a clearly defined target and separate training/test data.

### How this cell is different
- This cell prepares the complete dataset rather than training a model.
- The dataset is intentionally kept small so later AutoGluon steps run quickly in Colab.

### What this cell does
- Installs only **AutoGluon Tabular with LightGBM**, instead of the full AutoGluon package.
- Creates a synthetic household-budget dataset with numeric, Boolean, and categorical features.
- Adds a small amount of missing data.
- Creates the target `overspent_next`.
- Splits the data into training and test sets using `stratify`.

### What we achieve after running it
- `train_df` is ready for model training.
- `test_df` remains unseen for final evaluation.
- We can inspect the positive-class distribution before modeling.

### Remember
**Training set = model learns from it.**

**Test set = final unseen evaluation.**

The test set should not be used to train or tune the model.


In [ ]:
!pip install -q "autogluon.tabular[lightgbm]"

import os
import tempfile
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

from autogluon.tabular import TabularPredictor

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
n = 800

city_tier = rng.choice(["metro", "city", "town"], n, p=[0.35, 0.40, 0.25])
household_size = rng.integers(1, 6, n)
owns_car = rng.random(n) < 0.55
income = rng.lognormal(mean=8.4, sigma=0.35, size=n)

rent_ratio = pd.Series(city_tier).map({
    "metro": 0.34,
    "city": 0.27,
    "town": 0.20
}).to_numpy()

rent = income * rent_ratio * rng.normal(1.0, 0.05, n)
groceries = (180 + 95 * household_size) * rng.normal(1.0, 0.12, n)
utilities = (60 + 25 * household_size) * rng.normal(1.0, 0.10, n)
transport = np.where(owns_car, 210, 90) * rng.normal(1.0, 0.15, n)
dining = rng.gamma(2.0, 70, n)
entertainment = rng.gamma(1.8, 60, n) + 0.35 * dining
other = rng.gamma(1.5, 70, n)

total_spend = rent + groceries + utilities + transport + dining + entertainment + other

# Historical spending features
avg_spend_prev = total_spend * rng.normal(0.97, 0.08, n)
overspend_rate_prev = np.clip(
    (total_spend / income - 0.45) + rng.normal(0, 0.08, n),
    0, 1
)
max_dining_prev = dining * rng.uniform(1.0, 1.8, n)

# Create next-month overspending probability
logit = (
    -3.2
    + 4.5 * overspend_rate_prev
    + 2.2 * (total_spend / income)
    + 0.35 * (city_tier == "metro")
    + 0.20 * household_size
)

p_overspend = 1 / (1 + np.exp(-logit))
overspent_next = (rng.random(n) < p_overspend).astype(int)

df = pd.DataFrame({
    "city_tier": city_tier,
    "household_size": household_size,
    "owns_car": owns_car,
    "income": income.round(0),
    "rent": rent.round(0),
    "groceries": groceries.round(0),
    "utilities": utilities.round(0),
    "transport": transport.round(0),
    "dining": dining.round(0),
    "entertainment": entertainment.round(0),
    "other": other.round(0),
    "avg_spend_prev": avg_spend_prev.round(0),
    "overspend_rate_prev": overspend_rate_prev.round(3),
    "max_dining_prev": max_dining_prev.round(0),
    "survey_score": rng.integers(1, 6, n),
    "referral_code": rng.choice(["A1", "B2", "C3", None], n, p=[0.3, 0.3, 0.3, 0.1]),
    "overspent_next": overspent_next
})

# Add a little missingness
missing_rows = rng.random(n) < 0.04
df.loc[missing_rows, "dining"] = np.nan

LABEL = "overspent_next"

train_df, test_df = train_test_split(
    df,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=df[LABEL]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Full dataset:", df.shape)
print("Train:", train_df.shape, "| Test:", test_df.shape)
print("Positive rate:")
print(df[LABEL].value_counts(normalize=True).sort_index().round(3))
display(train_df.head())


## 2. Create a Baseline

### What we saw before
- We created separate training and test datasets.
- The test set is being kept separate from model training.

### How this cell is different
- Before using AutoML, we first train a simpler model.
- This gives us a **baseline** that AutoGluon should try to improve upon.

### What this cell does
- Uses numeric and Boolean features for a simple sklearn baseline.
- Fills missing numeric values using training-set medians.
- Trains `HistGradientBoostingClassifier`.
- Uses `predict_proba()` to calculate overspending probabilities.
- Calculates ROC-AUC.

### What we achieve after running it
- We get a reference ROC-AUC score.
- Later we can compare AutoGluon's performance against this baseline.

### Remember
Never judge an advanced model without a comparison.

**Baseline first → advanced model second.**


In [ ]:
numeric_features = [
    c for c in train_df.columns
    if c != LABEL and (
        pd.api.types.is_numeric_dtype(train_df[c])
        or pd.api.types.is_bool_dtype(train_df[c])
    )
]

X_train_base = train_df[numeric_features].copy()
X_test_base = test_df[numeric_features].copy()

# Fill missing numeric values for this sklearn baseline
medians = X_train_base.median()
X_train_base = X_train_base.fillna(medians)
X_test_base = X_test_base.fillna(medians)

baseline = HistGradientBoostingClassifier(
    random_state=RANDOM_STATE
)

baseline.fit(
    X_train_base,
    train_df[LABEL]
)

baseline_prob = baseline.predict_proba(X_test_base)[:, 1]
baseline_pred = (baseline_prob >= 0.5).astype(int)

baseline_auc = roc_auc_score(
    test_df[LABEL],
    baseline_prob
)

print("Baseline numeric features:", len(numeric_features))
print("Baseline ROC-AUC:", round(baseline_auc, 4))


## 3. Train AutoGluon

### What we saw before
- The baseline used one manually selected sklearn algorithm.
- We now have a reference ROC-AUC score.

### How this cell is different
- AutoGluon handles the full tabular dataset, including categorical columns and missing values.
- It compares more than one model family automatically.

### What this cell does
- `TabularPredictor(label=LABEL)` defines the target.
- `eval_metric="roc_auc"` defines the model-selection metric.
- We explicitly train only **LightGBM (`GBM`) and Random Forest (`RF`)**.
- Restricting the model set keeps the run fast while still giving us a meaningful leaderboard comparison.

### What we achieve after running it
- We get two AutoGluon model families trained on the same data.
- `model_best` identifies the strongest model based on validation performance.

### Remember
The important AutoGluon pattern is:

```python
predictor = TabularPredictor(
    label=target,
    eval_metric=metric
).fit(
    train_data,
    hyperparameters={...}
)
```

AutoGluon automates model fitting and comparison, but we still choose the **target, metric, split, and model budget**.


In [ ]:
MODEL_DIR = tempfile.mkdtemp(prefix="ag_part3_")

predictor = TabularPredictor(
    label=LABEL,
    eval_metric="roc_auc",
    path=MODEL_DIR,
    verbosity=0
).fit(
    train_df,
    hyperparameters={
        "GBM": {},
        "RF": {}
    }
)

print("Problem type:", predictor.problem_type)
print("Evaluation metric:", predictor.eval_metric.name)
print("Models trained:", len(predictor.model_names()))
print("Best model:", predictor.model_best)
print("Model directory:", MODEL_DIR)


## 4. Read the Leaderboard

### What we saw before
- AutoGluon trained LightGBM and Random Forest.
- `model_best` tells us which model won on AutoGluon's internal validation data.

### How this cell is different
- Instead of looking only at the winner, we compare the trained models side by side.
- We also evaluate them on our separate test set.

### What this cell does
- `leaderboard(test_df)` creates one row per trained model.
- `score_val` is the validation score used during model selection.
- `score_test` is performance on the held-out test data.
- `fit_time` shows training cost.
- `pred_time_test` shows prediction cost.
- The best AutoGluon test result is compared with the sklearn baseline.

### What we achieve after running it
- We can compare model quality and runtime.
- We can check whether the validation winner also performs well on unseen data.

### Remember
**Validation score = used for model selection.**

**Test score = independent final check.**


In [ ]:
leaderboard = predictor.leaderboard(
    test_df,
    silent=True
)

cols = [
    c for c in [
        "model",
        "score_test",
        "score_val",
        "fit_time",
        "pred_time_test",
        "stack_level"
    ]
    if c in leaderboard.columns
]

display(leaderboard[cols].head(10))

print("AutoGluon best model:", predictor.model_best)
print("Best test ROC-AUC:", round(leaderboard["score_test"].max(), 4))
print("Baseline ROC-AUC:", round(baseline_auc, 4))


## 5. Evaluate Predictions with Multiple Metrics

### What we saw before
- The leaderboard mainly compared models using ROC-AUC.
- One metric alone does not describe every type of model error.

### How this cell is different
- Now we inspect the predictions from the selected model in more detail.

### What this cell does
- `predict_proba()` returns positive-class probabilities.
- `predict()` returns final class labels.
- Calculates:
  - Accuracy
  - Balanced Accuracy
  - Precision
  - Recall
  - F1
  - ROC-AUC
  - PR-AUC
- Builds a confusion matrix.
- Displays a few example household predictions.

### What we achieve after running it
- We get a more complete picture of model quality.
- We can see both numerical metrics and actual examples.

### Remember
**Precision:** when we predict overspending, how often are we right?

**Recall:** how many actual overspenders did we find?

Different business problems care about different mistakes.


In [ ]:
y_true = test_df[LABEL].to_numpy()

y_prob = predictor.predict_proba(
    test_df.drop(columns=[LABEL])
)[1].to_numpy()

y_pred = predictor.predict(
    test_df.drop(columns=[LABEL])
).to_numpy()

metrics = pd.DataFrame([{
    "accuracy": accuracy_score(y_true, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, zero_division=0),
    "recall": recall_score(y_true, y_pred, zero_division=0),
    "f1": f1_score(y_true, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_true, y_prob),
    "pr_auc": average_precision_score(y_true, y_prob),
}])

display(metrics.round(4))

cm = confusion_matrix(y_true, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)

print("Confusion matrix:")
display(cm_df)

sample_predictions = test_df.head(5).copy()
sample_predictions["P(overspend)"] = y_prob[:5].round(3)
sample_predictions["prediction"] = y_pred[:5]

display(
    sample_predictions[
        ["city_tier", "income", "overspend_rate_prev",
         "P(overspend)", "prediction", LABEL]
    ]
)


## 6. Decision Thresholds

### What we saw before
- The model gives a probability for each household.
- The normal prediction converts that probability into a class.

### How this cell is different
- We do not retrain the model.
- We only change the cutoff used to convert probability into an action.

### What this cell does
- Tests thresholds `0.30`, `0.50`, and `0.70`.
- Compares Precision, Recall, F1, and predicted-positive rate.
- Shows how a lower or higher threshold changes model behavior.

### What we achieve after running it
- We see that model probability and business decision are separate concepts.
- We can observe the Precision/Recall trade-off.

### Remember
**Model → produces probability.**

**Threshold → converts probability into an action.**

The best threshold depends on the business cost of false positives and false negatives.


In [ ]:
rows = []

for threshold in [0.30, 0.50, 0.70]:
    pred_t = (y_prob >= threshold).astype(int)

    rows.append({
        "threshold": threshold,
        "precision": precision_score(y_true, pred_t, zero_division=0),
        "recall": recall_score(y_true, pred_t, zero_division=0),
        "f1": f1_score(y_true, pred_t, zero_division=0),
        "predicted_positive_rate": pred_t.mean()
    })

threshold_table = pd.DataFrame(rows)

display(threshold_table.round(3))


## 7. Feature Importance

### What we saw before
- We know how well the model performs.
- We have also seen how decision thresholds change predictions.

### How this cell is different
- This section focuses on **why the model makes its predictions** rather than only on performance.

### What this cell does
- Uses AutoGluon's permutation feature importance.
- A feature is shuffled and the change in model performance is measured.
- To keep this step fast, only 200 test rows and two shuffle sets are used.

### What we achieve after running it
- We identify the features the selected model relies on most.
- The output provides importance, variation, and statistical information.

### Remember
Permutation importance asks:

**"How much worse does the model become when this feature is disrupted?"**

Feature importance describes **model dependence**, not causation.


In [ ]:
importance = predictor.feature_importance(
    test_df,
    subsample_size=200,
    num_shuffle_sets=2,
    silent=True
)

display(
    importance[
        ["importance", "stddev", "p_value"]
    ].head(10).round(4)
)

print(
    "Top 3 features:",
    list(importance.head(3).index)
)


## 8. Final End-to-End Takeaways

### What we saw before
- We prepared data and created an honest train/test split.
- We created a baseline.
- AutoGluon compared LightGBM and Random Forest.
- The leaderboard compared validation and test performance.
- Multiple metrics evaluated the selected model.
- Thresholds demonstrated the Precision/Recall trade-off.
- Feature importance showed which inputs the model relied on.

### Complete workflow

**1. Define the target**

`overspent_next`

↓

**2. Split training and test data**

↓

**3. Build a baseline**

↓

**4. Train AutoGluon**

```python
TabularPredictor(...).fit(train_df)
```

↓

**5. Compare models**

```python
predictor.leaderboard(test_df)
```

↓

**6. Generate predictions**

```python
predictor.predict_proba(...)
predictor.predict(...)
```

↓

**7. Evaluate with multiple metrics**

Accuracy, Precision, Recall, F1, ROC-AUC, PR-AUC

↓

**8. Explore decision thresholds**

↓

**9. Inspect feature importance**

### Remember
- AutoML reduces model-training boilerplate, but it does not replace ML judgment.
- `score_val` is for model selection; the held-out test set is for final evaluation.
- Accuracy alone can hide important class-specific errors.
- Probabilities are useful because business thresholds can be changed without retraining.
- Feature importance explains model dependence, not causality.
